In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
ALPHA = 0.01
ITERATIONS = 1000
K = 5

DATA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

COLUMNS = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df = pd.read_csv(DATA_URL, header=None, names=COLUMNS)

print("Shape:", df.shape)
display(df.head())


In [ ]:
invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

zero_counts = (df[invalid_zero_columns] == 0).sum().sort_values(ascending=False)

print("Invalid zero counts:")
display(zero_counts.to_frame("Zero_Count"))

print("\nTotal invalid zero values:", int(zero_counts.sum()))


In [ ]:
X = df.drop(columns="Outcome").copy()
y = df["Outcome"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)

print("\nTraining class distribution:")
display(y_train.value_counts(normalize=True).rename("proportion"))

print("\nTesting class distribution:")
display(y_test.value_counts(normalize=True).rename("proportion"))


In [ ]:
def fit_outcome_conditioned_medians(X_train, y_train, columns):
    medians = {}
    for col in columns:
        medians[col] = {}
        for outcome_value in [0, 1]:
            valid_values = X_train.loc[
                (y_train == outcome_value) & (X_train[col] != 0),
                col
            ]
            medians[col][outcome_value] = valid_values.median()
    return medians


def apply_outcome_conditioned_imputation(X_data, y_data, medians, columns):
    X_clean = X_data.copy()

    for col in columns:
        for outcome_value in [0, 1]:
            mask = (y_data == outcome_value) & (X_clean[col] == 0)
            X_clean.loc[mask, col] = medians[col][outcome_value]

    return X_clean


train_medians = fit_outcome_conditioned_medians(
    X_train,
    y_train,
    invalid_zero_columns
)

print("Training medians used for imputation:")
median_table = pd.DataFrame(train_medians).T
display(median_table)

X_train_clean = apply_outcome_conditioned_imputation(
    X_train,
    y_train,
    train_medians,
    invalid_zero_columns
)

X_test_clean = apply_outcome_conditioned_imputation(
    X_test,
    y_test,
    train_medians,
    invalid_zero_columns
)

print("Remaining invalid zeros in training data:")
display((X_train_clean[invalid_zero_columns] == 0).sum())

print("Remaining invalid zeros in test data:")
display((X_test_clean[invalid_zero_columns] == 0).sum())


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)


In [ ]:
class CustomLogisticRegression:
    def __init__(self, alpha=0.01, iterations=1000):
        self.alpha = alpha
        self.iterations = iterations
        self.weights = None
        self.bias = None
        self.loss_history = []

    @staticmethod
    def sigmoid(z):
        # Clip values to prevent numerical overflow in exp()
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def compute_loss(self, y, probabilities):
        eps = 1e-15
        probabilities = np.clip(probabilities, eps, 1 - eps)

        return -np.mean(
            y * np.log(probabilities)
            + (1 - y) * np.log(1 - probabilities)
        )

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []

        for _ in range(self.iterations):
            # Forward pass
            linear_output = np.dot(X, self.weights) + self.bias
            probabilities = self.sigmoid(linear_output)

            # Loss
            loss = self.compute_loss(y, probabilities)
            self.loss_history.append(loss)

            # Gradients
            error = probabilities - y

            dw = (1 / n_samples) * np.dot(X.T, error)
            db = (1 / n_samples) * np.sum(error)

            # Gradient descent update
            self.weights -= self.alpha * dw
            self.bias -= self.alpha * db

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        linear_output = np.dot(X, self.weights) + self.bias
        probabilities = self.sigmoid(linear_output)

        return np.column_stack([
            1 - probabilities,
            probabilities
        ])

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= threshold).astype(int)


custom_lr = CustomLogisticRegression(
    alpha=ALPHA,
    iterations=ITERATIONS
)

custom_lr.fit(X_train_scaled, y_train)

print("Final training loss:", custom_lr.loss_history[-1])


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, ITERATIONS + 1), custom_lr.loss_history)
plt.xlabel("Iteration")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Custom Logistic Regression — Training Loss")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
lr_pred_05 = custom_lr.predict(X_test_scaled, threshold=0.50)

lr_accuracy = accuracy_score(y_test, lr_pred_05)
lr_f1 = f1_score(y_test, lr_pred_05)
lr_precision = precision_score(y_test, lr_pred_05, zero_division=0)
lr_recall = recall_score(y_test, lr_pred_05)

print("Custom Logistic Regression @ threshold 0.50")
print(f"Accuracy : {lr_accuracy:.4f}")
print(f"F1-score : {lr_f1:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_pred_05))


In [ ]:
knn_raw = KNeighborsClassifier(n_neighbors=K)
knn_scaled = KNeighborsClassifier(n_neighbors=K)

knn_raw.fit(X_train_clean, y_train)
knn_scaled.fit(X_train_scaled, y_train)

knn_raw_pred = knn_raw.predict(X_test_clean)
knn_scaled_pred = knn_scaled.predict(X_test_scaled)

knn_results = pd.DataFrame({
    "Model": [
        "KNN - Unscaled",
        "KNN - Standardized"
    ],
    "Accuracy": [
        accuracy_score(y_test, knn_raw_pred),
        accuracy_score(y_test, knn_scaled_pred)
    ],
    "F1-score": [
        f1_score(y_test, knn_raw_pred),
        f1_score(y_test, knn_scaled_pred)
    ]
})

display(knn_results)


In [ ]:
test_probabilities = custom_lr.predict_proba(X_test_scaled)[:, 1]

thresholds = np.arange(0.20, 0.81, 0.05)

threshold_rows = []

for threshold in thresholds:
    preds = (test_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        preds,
        labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "Threshold": round(float(threshold), 2),
        "Accuracy": accuracy_score(y_test, preds),
        "F1": f1_score(y_test, preds, zero_division=0),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall/Sensitivity": recall_score(y_test, preds, zero_division=0),
        "False Negatives": fn,
        "False Positives": fp
    })

threshold_results = pd.DataFrame(threshold_rows)

display(threshold_results.round(4))


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Recall/Sensitivity"],
    marker="o",
    label="Recall / Sensitivity"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    marker="o",
    label="F1-score"
)

plt.xlabel("Decision Threshold")
plt.ylabel("Metric")
plt.title("Clinical Threshold Trade-off")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
TARGET_SENSITIVITY = 0.90

eligible = threshold_results[
    threshold_results["Recall/Sensitivity"] >= TARGET_SENSITIVITY
].copy()

if len(eligible) > 0:
    selected_threshold = eligible["Threshold"].max()
else:
    # If no tested threshold reaches the target, choose the threshold
    # with the highest sensitivity.
    selected_threshold = threshold_results.loc[
        threshold_results["Recall/Sensitivity"].idxmax(),
        "Threshold"
    ]

selected_pred = (test_probabilities >= selected_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    selected_pred,
    labels=[0, 1]
).ravel()

print(f"Selected threshold: {selected_threshold:.2f}")
print(f"Accuracy : {accuracy_score(y_test, selected_pred):.4f}")
print(f"F1-score : {f1_score(y_test, selected_pred, zero_division=0):.4f}")
print(f"Precision: {precision_score(y_test, selected_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, selected_pred, zero_division=0):.4f}")
print(f"False negatives: {fn}")
print(f"False positives: {fp}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, selected_pred))


In [ ]:
final_results = pd.DataFrame({
    "Model": [
        "Custom Logistic Regression (scaled, threshold=0.50)",
        f"Custom Logistic Regression (scaled, threshold={selected_threshold:.2f})",
        "KNN (unscaled)",
        "KNN (standardized)"
    ],
    "Accuracy": [
        accuracy_score(y_test, lr_pred_05),
        accuracy_score(y_test, selected_pred),
        accuracy_score(y_test, knn_raw_pred),
        accuracy_score(y_test, knn_scaled_pred)
    ],
    "F1-score": [
        f1_score(y_test, lr_pred_05, zero_division=0),
        f1_score(y_test, selected_pred, zero_division=0),
        f1_score(y_test, knn_raw_pred, zero_division=0),
        f1_score(y_test, knn_scaled_pred, zero_division=0)
    ],
    "Precision": [
        precision_score(y_test, lr_pred_05, zero_division=0),
        precision_score(y_test, selected_pred, zero_division=0),
        precision_score(y_test, knn_raw_pred, zero_division=0),
        precision_score(y_test, knn_scaled_pred, zero_division=0)
    ],
    "Recall/Sensitivity": [
        recall_score(y_test, lr_pred_05, zero_division=0),
        recall_score(y_test, selected_pred, zero_division=0),
        recall_score(y_test, knn_raw_pred, zero_division=0),
        recall_score(y_test, knn_scaled_pred, zero_division=0)
    ]
})

display(final_results.round(4))
